# 13.3 포스트트레이닝: SFT, RLHF, DPO — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter13_3_posttraining_alignment.ipynb)

책 본문: [13.3 포스트트레이닝: SFT, RLHF, DPO](https://smhanlab.com/book-ml/kor/ml1/chapter13/3.html)

이 노트북은 책 13.3절의 모든 수치를 **실제로 실행해서** 검증합니다:

1. **BT 손실 & 그래디언트** — \(r_w=1, r_l=-1 \Rightarrow 0.1269\), 거꾸로 \(2.1269\), margin 0 \(\Rightarrow \log 2 \approx 0.693\)
2. **장난감 보상모델 학습** — 2차원 특징(길이, 거절 여부) 5쌍을 SGD로 분류: 손실 \(3.47 \to 0.002\)
3. **KL & 보상 해킹** — \(D_{KL}((0.95,0.05)\|(0.75,0.25)) \approx 0.495\); \(\beta = 0,1,10,100\)의 \(q^*\)·\(V(q^*)\)
4. **DPO 손실** — 초기 \(z \approx 0.811\)(손실 0.368) → 수렴 \(z \approx 1.658\)(손실 0.174)
5. **LoRA/SVD** — rank-4 \(\Delta W\)를 rank 4/3/1로 근사: 상대 오차 0 / 0.326 / 0.701
6. **SFT 분포 재조정** — 7단계 이상적 답변: \(P_{\text{ref}} \approx 0.0011\) vs \(P_{\text{SFT}} \approx 0.0323\)

In [1]:
import math

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)

numpy 2.4.6


## 1. BT(브래들리-테리) 손실 & 그래디언트

본문의 `reward_model_loss`를 그대로 실행합니다. 손실은 **margin
\(z = r_w - r_l\)의 함수**이고, 그래디언트
\(\frac{d\mathcal{L}}{dz} = \sigma(z) - 1\)가
"(예측 − 정답)" 꼴(Chapter 2 로지스틱회귀와 동일)이라는 것을
확인합니다 — 정답 라벨이 항상 1이므로.

- \(z = 2\) (잘 구분) → 손실 ≈ 0.1269, 그래디언트 ≈ −0.119 (거의 0, 갱신 거의 안 됨)
- \(z = -2\) (거꾸로) → 손실 ≈ 2.1269, 그래디언트 ≈ −0.881 (강하게 교정)
- \(z = 0\) (무작위) → 손실 = \(\log 2 \approx 0.693\), 그래디언트 = −0.5

In [2]:
def sigmoid(z):
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    ez = math.exp(z)
    return ez / (1 + ez)

def reward_model_loss(r_win, r_lose):
    # Bradley-Terry: P(y_w > y_l) = sigmoid(r_win - r_lose)
    return -math.log(sigmoid(r_win - r_lose))

def rm_grad_margin(r_win, r_lose):
    # dL/dz = sigmoid(z) - 1  (로지스틱회귀 '예측 - 정답(=1)'과 동일)
    return sigmoid(r_win - r_lose) - 1

# 잘 구분한 쌍 (margin = 2)
print(f"r_w=1, r_l=-1 (z=+2):  loss = {reward_model_loss(1.0, -1.0):.4f}   grad = {rm_grad_margin(1.0, -1.0):.4f}")
assert abs(reward_model_loss(1.0, -1.0) - 0.1269) < 5e-5
# 점수가 거꾸로인 쌍 (margin = -2)
print(f"r_w=-1, r_l=1 (z=-2):  loss = {reward_model_loss(-1.0, 1.0):.4f}   grad = {rm_grad_margin(-1.0, 1.0):.4f}")
assert abs(reward_model_loss(-1.0, 1.0) - 2.1269) < 5e-5
# 동전 던지기 (margin = 0)
print(f"r_w=0, r_l=0  (z= 0):  loss = {reward_model_loss(0.0, 0.0):.4f}   grad = {rm_grad_margin(0.0, 0.0):.4f}")
assert abs(reward_model_loss(0.0, 0.0) - math.log(2)) < 1e-12
# 본문 연습문제 1의 예: r_w=2, r_l=1 (margin 1)
print(f"r_w=2, r_l=1  (z=+1):  loss = {reward_model_loss(2.0, 1.0):.4f}")
assert abs(reward_model_loss(2.0, 1.0) - 0.3133) < 5e-5
# 기존 본문 값 재확인
assert abs(reward_model_loss(2.0, -1.0) - 0.049) < 1e-3
assert abs(reward_model_loss(-1.0, 2.0) - 3.049) < 1e-3
# '모델이 y_w를 확신하면 그래디언트 -> 0'
print(f"z=5(확신): grad = {rm_grad_margin(2.5, -2.5):.6f}  (0에 수렴 -- 더 이상 크게 갱신하지 않음)")
assert abs(rm_grad_margin(2.5, -2.5)) < 1e-2
print("BT 손실·그래디언트 확인 완료")

r_w=1, r_l=-1 (z=+2):  loss = 0.1269   grad = -0.1192
r_w=-1, r_l=1 (z=-2):  loss = 2.1269   grad = -0.8808
r_w=0, r_l=0  (z= 0):  loss = 0.6931   grad = -0.5000
r_w=2, r_l=1  (z=+1):  loss = 0.3133
z=5(확신): grad = -0.006693  (0에 수렴 -- 더 이상 크게 갱신하지 않음)
BT 손실·그래디언트 확인 완료


## 2. 장난감 보상모델 학습 (SGD로, 로지스틱회귀 기계 재사용)

본문의 5쌍 데이터 — winner는 항상 더 **짧고** 거절하지 않는 쪽.
특징 = (길이, 거절 여부), 보상모델 \(r = w_1 \cdot \text{길이}
+ w_2 \cdot \text{거절 여부}\). BT 손실이 로지스틱회귀 손실과
같은 모양이므로 **SGD 구현을 그대로** 씁니다 (\(\eta = 0.5\),
300 epoch, 배치=5쌍 전체).

기대: 손실 \(5\log 2 \approx 3.47\)에서 \(\approx 0.002\)
으로, 5/5 정확 분류, \(w \approx (-1.90, -6.69)\)
(둘 다 음수 = "짧고, 거절 안 하면 좋음").

In [3]:
# (winner, loser) — 각 (길이, 거절여부)
pairs = [
    ((2, 0), (5, 0)),
    ((3, 0), (7, 1)),
    ((4, 0), (8, 2)),
    ((2, 0), (3, 1)),
    ((5, 0), (4, 1)),
]

def rm(f, w):
    return w[0] * f[0] + w[1] * f[1]

w = [0.0, 0.0]
eta = 0.5
def mean_loss(w):
    return sum(reward_model_loss(rm(fw, w), rm(fl, w)) for fw, fl in pairs) / len(pairs)
losses = [mean_loss(w)]          # epoch 0 (갱신 전): 모든 margin = 0
for epoch in range(300):
    grad = [0.0, 0.0]
    for fw, fl in pairs:
        m = rm(fw, w) - rm(fl, w)
        g = sigmoid(m) - 1            # (예측 - 정답), dL/dw = g * dm/dw
        grad[0] += g * (fw[0] - fl[0])
        grad[1] += g * (fw[1] - fl[1])
    w = [w[i] - eta * grad[i] for i in range(2)]
    losses.append(mean_loss(w))

acc = sum(rm(fw, w) > rm(fl, w) for fw, fl in pairs)
print(f"초기 손실: 합 = 5*log2 = {5*math.log(2):.4f},  쌍당 평균 = log2 = {math.log(2):.4f}")
print(f"평균 손실  epoch 0(갱신 전): {losses[0]:.4f}   epoch 99: {losses[99]:.5f}   epoch 299: {losses[299]:.5f}")
print(f"w = ({w[0]:.4f}, {w[1]:.4f})   정확도 = {acc}/5")
assert abs(losses[0] - math.log(2)) < 1e-9
assert losses[299] < 0.01
assert acc == 5
assert w[0] < 0 and w[1] < 0
print(f"|w2/w1| = {abs(w[1]/w[0]):.2f}  ('거절' 스케일이 '길이'보다 작으므로 더 큰 가중치)")
print("장난감 보상모델: 5/5 구분, 손실 ~0.002 -- 확인 완료")

초기 손실: 합 = 5*log2 = 3.4657,  쌍당 평균 = log2 = 0.6931
평균 손실  epoch 0(갱신 전): 0.6931   epoch 99: 0.00665   epoch 299: 0.00238
w = (-1.8956, -6.6854)   정확도 = 5/5
|w2/w1| = 3.53  ('거절' 스케일이 '길이'보다 작으므로 더 큰 가중치)
장난감 보상모델: 5/5 구분, 손실 ~0.002 -- 확인 완료


## 3. KL 발산 & 보상 해킹

**(a) KL 수치 감각.** \(\pi_{\text{ref}} = (0.5, 0.5)\) 기준에서,
동일 정책의 KL은 0이고, 극단적 정책 \((0.95, 0.05)\)의 KL은
≈ 0.495입니다.

**(b) 보상 해킹 1차원 문제.** 품질 \(q \in [0,2]\),
진짜 가치 \(V(q) = 2q - 3q^2\), 보상모델 근사
\(r(q) = 2q - 0.2q^2\)(고품질 구간에서 하락을 과소평가),
\(\pi_{\text{ref}}\)의 KL ≈ \(\tfrac{1}{2}(q-0.5)^2\).
각 \(\beta\)에서 \(q^* = \arg\max_q [r(q) - \beta \cdot
\tfrac{1}{2}(q-0.5)^2]\)를 구해 봅니다 — \(\beta = 0\)에서
\(q^* = 2.00, V = -8\)(보상 해킹)이 나와야 합니다.

In [4]:
def kl_two(p, q):
    return sum(pi * math.log(pi / qi) for pi, qi in zip(p, q) if pi > 0)

print("(a) KL 수치 감각:")
print(f"  KL((0.95,0.05) | (0.5,0.5)) = {kl_two((0.95, 0.05), (0.5, 0.5)):.4f}")
assert abs(kl_two((0.95, 0.05), (0.5, 0.5)) - 0.495) < 5e-4
print(f"  KL((0.5,0.5) | (0.5,0.5)) = {kl_two((0.5, 0.5), (0.5, 0.5)):.6f}  (동일 -> 0)")
assert kl_two((0.5, 0.5), (0.5, 0.5)) == 0.0

# (b) 보상 해킹
def V_true(q):
    return 2 * q - 3 * q * q
def r_model(q):
    return 2 * q - 0.2 * q * q
def kl_of(q):
    return 0.5 * (q - 0.5) ** 2

qs = np.linspace(0.0, 2.0, 20001)
print("\n(b) 보상 해킹: beta별 최적 q* (V = 진짜 가치)")
print(f"{'beta':>6} {'q*':>7} {'r(q*)':>7} {'KL':>8} {'V(q*)':>8}  net r - beta*KL")
rows = []
for beta in [0, 1, 10, 100]:
    net = r_model(qs) - beta * kl_of(qs)
    qstar = qs[np.argmax(net)]
    rows.append((beta, qstar, r_model(qstar), kl_of(qstar), V_true(qstar)))
    print(f"{beta:>6} {qstar:>7.3f} {r_model(qstar):>7.3f} {kl_of(qstar):>8.4f} {V_true(qstar):>8.3f}   {net.max():.3f}")

assert rows[0][1] == 2.0 and abs(rows[0][4] - (-8.0)) < 1e-9   # beta=0: 경계까지, V=-8
assert abs(rows[2][1] - 0.673) < 5e-3                          # beta=10
assert rows[3][1] < 0.53                                        # beta=100: 거의 동결
qV = 1 / 3
print(f"\n참고: V의 진짜 최대값은 q=1/3에서 V = {V_true(qV):.4f}")
assert abs(V_true(qV) - 1 / 3) < 1e-12   # V(1/3) = 2/3 - 3/9 = 1/3

# 그래프
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
ax = axes[0]
ax.plot(qs, V_true(qs), color="tab:green", lw=2.2, label="True value $V(q) = 2q - 3q^2$")
ax.plot(qs, r_model(qs), color="tab:orange", lw=2.2, ls="--", label="Reward model $r(q) = 2q - 0.2q^2$")
for beta, c in zip([0, 1, 10, 100], ["tab:red", "tab:blue", "tab:purple", "gray"]):
    _, qstar, _, _, _ = rows[[b[0] for b in rows].index(beta)]
    ax.plot([qstar, qstar], [V_true(qstar), r_model(qstar)], color=c, lw=1.2, alpha=0.7)
    ax.scatter([qstar], [V_true(qstar)], color=c, zorder=5, s=45)
    ax.annotate(f"$\\beta$={beta}\n$q^*$={qstar:.2f}", (qstar, V_true(qstar)),
                textcoords="offset points", xytext=(8, 8), fontsize=8.5, color=c)
ax.set_xlabel("Quality $q$")
ax.set_ylabel("Value / Reward")
ax.set_title("Reward model underestimates V at high quality (extrapolation error)")
ax.legend(fontsize=8)
ax2 = axes[1]
for beta, c in zip([0, 1, 10, 100], ["tab:red", "tab:blue", "tab:purple", "gray"]):
    b2 = rows[[b[0] for b in rows].index(beta)]
    ax2.scatter([beta], [b2[4]], color=c, zorder=5, s=45)
ax2.plot([0, 100], [V_true(qV), V_true(qV)], color="tab:green", ls=":", lw=1.5)
ax2.text(1, V_true(qV) + 0.08, "True maximum $V(1/3) = +0.67$", fontsize=8, color="tab:green")
ax2.set_xscale("symlog", linthresh=1)   # beta=0(=없음)도 표시 가능
ax2.set_xticks([0, 1, 10, 100])
ax2.set_xlabel("KL penalty strength $\\beta$")
ax2.set_ylabel("$V(q^*(\\beta))$ -- true value")
ax2.set_title("$\\beta$=0: reward hacking (V=−8) / $\\beta$=100: frozen (V=+0.23)")
ax2.grid(alpha=0.3)
plt.tight_layout()
p = f"{IMG}/ch13_3_reward_hacking.svg"
plt.savefig(p, bbox_inches="tight")
plt.show()
print(f"SVG 저장: {p}")
print("-> beta가 작으면 보상 해킹, 크면 over-regularization. 확인 완료")

(a) KL 수치 감각:
  KL((0.95,0.05) | (0.5,0.5)) = 0.4946
  KL((0.5,0.5) | (0.5,0.5)) = 0.000000  (동일 -> 0)

(b) 보상 해킹: beta별 최적 q* (V = 진짜 가치)
  beta      q*   r(q*)       KL    V(q*)  net r - beta*KL
     0   2.000   3.200   1.1250   -8.000   3.200
     1   1.786   2.934   0.8265   -5.995   2.107
    10   0.673   1.256   0.0150   -0.013   1.106
   100   0.518   0.982   0.0002    0.231   0.966

참고: V의 진짜 최대값은 q=1/3에서 V = 0.3333


SVG 저장: /home/smhan/book-ml/kor/src/images/ch13_3_reward_hacking.svg
-> beta가 작으면 보상 해킹, 크면 over-regularization. 확인 완료


## 4. DPO 손실 — "보상 역할"을 정책 로그-비율이 겸임

본문 숫자 예(\(\beta = 1\)): 초기 정책
\(\pi_\theta(y_w|x)=0.6\), \(\pi_\theta(y_l|x)=0.2\), 기준
\(\pi_{\text{ref}}(y_w|x)=0.4\), \(\pi_{\text{ref}}(y_l|x)=0.3\).
DPO 손실은 BT 손실과 **같은 형태** — 다만 보상의 자리에
\(\beta \log(\pi_\theta/\pi_{\text{ref}})\)가 들어간다.

- 초기: margin \(z \approx 0.811\) → 손실 ≈ 0.368
- 학습 후 (0.7, 0.1): \(z \approx 1.658\) → 손실 ≈ 0.174
- 그래디언트 방향: \(y_w\) 로그-비율 **올리고**, \(y_l\) **내리기**
- \(\beta = 0.1\)이면 margin이 10분의 1로 축소

In [5]:
def dpo_loss(logp_win_theta, logp_win_ref, logp_lose_theta, logp_lose_ref, beta):
    log_ratio_win = beta * (logp_win_theta - logp_win_ref)
    log_ratio_lose = beta * (logp_lose_theta - logp_lose_ref)
    return -math.log(sigmoid(log_ratio_win - log_ratio_lose))

# 초기: (0.6, 0.2) vs ref (0.4, 0.3)
lrw1 = math.log(0.6 / 0.4)
lrl1 = math.log(0.2 / 0.3)
z1 = lrw1 - lrl1
print(f"초기:  log-ratio win = {lrw1:.4f},  lose = {lrl1:.4f}")
print(f"      margin z = {z1:.4f}   loss(beta=1) = {dpo_loss(math.log(0.6), math.log(0.4), math.log(0.2), math.log(0.3), 1.0):.4f}")
assert abs(z1 - 0.8109) < 5e-4
assert abs(dpo_loss(math.log(0.6), math.log(0.4), math.log(0.2), math.log(0.3), 1.0) - 0.3677) < 5e-4
g_win = sigmoid(z1) - 1      # dL/d(log_ratio_win)
g_lose = 1 - sigmoid(z1)     # dL/d(log_ratio_lose)
print(f"      grad: win {g_win:+.4f} (비율 올림),  lose {g_lose:+.4f} (비율 내림)")
assert g_win < 0 < g_lose

# 학습 후: (0.7, 0.1)
lrw2 = math.log(0.7 / 0.4)
lrl2 = math.log(0.1 / 0.3)
z2 = lrw2 - lrl2
print(f"학습 후: margin z = {z2:.4f}   loss = {dpo_loss(math.log(0.7), math.log(0.4), math.log(0.1), math.log(0.3), 1.0):.4f}")
assert abs(z2 - 1.6582) < 5e-4
assert dpo_loss(math.log(0.7), math.log(0.4), math.log(0.1), math.log(0.3), 1.0) < 0.2
# beta = 0.1: margin 축소
print(f"beta=0.1 초기: z = {0.1 * z1:.4f}   loss = {dpo_loss(math.log(0.6), math.log(0.4), math.log(0.2), math.log(0.3), 0.1):.4f}")
assert abs(0.1 * z1 - 0.0811) < 5e-4
# 확인문제 2: log ratios 1.0, -0.5 (beta=1 -> z=1.5; beta=0.5 -> z=0.75)
z_15 = 1.0 - (-0.5)
print(f"확인문제2: beta=1:  z={z_15:.2f}, loss={-math.log(sigmoid(z_15)):.4f} (sigma={sigmoid(z_15):.4f})")
print(f"           beta=0.5: z={0.5 * z_15:.2f}, loss={-math.log(sigmoid(0.5 * z_15)):.4f}")
assert abs(-math.log(sigmoid(z_15)) - 0.2014) < 5e-4   # -log sigmoid(1.5)
assert abs(-math.log(sigmoid(0.5 * z_15)) - 0.3869) < 5e-4
print("DPO 손실: BT 손실과 같은 곡선, 보상의 자리에 정책 로그-비율. 확인 완료")

초기:  log-ratio win = 0.4055,  lose = -0.4055
      margin z = 0.8109   loss(beta=1) = 0.3677
      grad: win -0.3077 (비율 올림),  lose +0.3077 (비율 내림)
학습 후: margin z = 1.6582   loss = 0.1744
beta=0.1 초기: z = 0.0811   loss = 0.6534
확인문제2: beta=1:  z=1.50, loss=0.2014 (sigma=0.8176)
           beta=0.5: z=0.75, loss=0.3869
DPO 손실: BT 손실과 같은 곡선, 보상의 자리에 정책 로그-비율. 확인 완료


## 5. LoRA: "저차원"이 왜 충분한가 (SVD로 검증)

LoRA의 가설 — 파인튜닝의 **효과적 변화** \(\Delta W\)가 본래
\(W\)보다 **낮은 rank**를 가진다. \(20 \times 20\) 행렬에
rank-4인 \(\Delta W\)를 만들어 SVD로 rank 4/3/1 근사의 상대
오차를 재봅니다 — rank 4에서 오차 0(정확 재현), rank를 떨어뜨릴수록
급격히 커져야 합니다. 마지막에 실제 LLM 규모
(\(4096 \times 4096\), \(r = 8\))의 파라미터 비율도 계산합니다.

In [6]:
np.random.seed(0)
W = np.random.randn(20, 20)                    # 원래 가중치 (고정)
dW = np.random.randn(20, 4) @ np.random.randn(4, 20)   # rank-4 '학습된 변화'
print(f"rank(dW) = {np.linalg.matrix_rank(dW)}   (4로 생성)")

U, s, Vt = np.linalg.svd(dW)
print(f"\n{'rank r':>7} {'파라미터 (vs 400)':>18} {'상대 오차 ||dW - appr||/||dW||':>30}")
for r in [4, 3, 1]:
    appr = (U[:, :r] * s[:r]) @ Vt[:r]         # rank-r 근사
    rel = np.linalg.norm(dW - appr) / np.linalg.norm(dW)
    print(f"{r:>7} {2 * 20 * r:>10} ({2 * 20 * r / 400:4.0%})   {rel:>24.3e}")
assert np.linalg.matrix_rank(dW) == 4
assert np.linalg.norm(dW - (U[:, :4] * s[:4]) @ Vt[:4]) < 1e-10   # rank-4는 정확

# 실제 LLM 규모: 4096x4096 한 층, r=8
full = 4096 * 4096
lora = 2 * 4096 * 8
print(f"\n4096x4096 층 (r=8): 전체 {full:,}  LoRA {lora:,}  = {lora / full:.3%}")
assert abs(lora / full - 0.00391) < 1e-4
print("LoRA: 저차원 보정만으로 효율. 확인 완료")

rank(dW) = 4   (4로 생성)

 rank r      파라미터 (vs 400)     상대 오차 ||dW - appr||/||dW||
      4        160 ( 40%)                  1.058e-15
      3        120 ( 30%)                  3.260e-01
      1         40 ( 10%)                  7.014e-01

4096x4096 층 (r=8): 전체 16,777,216  LoRA 65,536  = 0.391%
LoRA: 저차원 보정만으로 효율. 확인 완료


## 6. SFT 분포 재조정 — "곱"이 격차를 만든다

본문 예 — 7단계 이상적 답변에 대해, ref 모델은 각 단계 정답 확률
\((0.2, 0.5, 0.3, 0.4, 0.3, 0.5, 0.6)\), SFT 모델은
\((0.8, 0.7, 0.5, 0.6, 0.4, 0.6, 0.8)\). 13.1절의 사슬 분해(= 각
단계 확률의 곱)로 **답변 전체** 확률과 평균 \(-\log p\)를
계산합니다 — "단계별 2~4배"가 전체로는 **약 30배**가 됩니다.

In [7]:
pref = [0.2, 0.5, 0.3, 0.4, 0.3, 0.5, 0.6]   # SFT 이전 (ref)
sftp = [0.8, 0.7, 0.5, 0.6, 0.4, 0.6, 0.8]   # SFT 이후

p_ref = math.prod(pref)
p_sft = math.prod(sftp)
ce_ref = -math.log(p_ref) / len(pref)
ce_sft = -math.log(p_sft) / len(sftp)
print(f"P_ref(y*)  = {p_ref:.6f}   (평균 -log p = {ce_ref:.4f})")
print(f"P_SFT(y*) = {p_sft:.6f}   (평균 -log p = {ce_sft:.4f})")
print(f"비율 P_SFT / P_ref = {p_sft / p_ref:.1f}x")
assert abs(p_ref - 0.00108) < 5e-6
assert abs(p_sft - 0.032256) < 5e-6
assert 25 < p_sft / p_ref < 35
print("-> 단계별 확률 2~4배 -> 전체(곱) 약 30배. SFT = '선택 기준의 재조정'. 확인 완료")

P_ref(y*)  = 0.001080   (평균 -log p = 0.9758)
P_SFT(y*) = 0.032256   (평균 -log p = 0.4906)
비율 P_SFT / P_ref = 29.9x
-> 단계별 확률 2~4배 -> 전체(곱) 약 30배. SFT = '선택 기준의 재조정'. 확인 완료


## 7. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| BT 손실 (z=+2 / −2 / 0) | 0.1269 / 2.1269 / 0.693 | 로지스틱회귀와 같은 \(-\log\sigma\) 형태; margin 0 = 무작위 |
| BT 그래디언트 | \(\sigma(z)-1\), z=5에서 ≈0 | "(예측 − 정답)" 꼴; 확신되면 갱신 멈춤 |
| 장난감 RM 5쌍 SGD | 3.47 → 0.002, 5/5, \(w\approx(-1.90,-6.69)\) | 보상모델 = 로지스틱회귀 기계 재사용; '짧고, 거절 안 하면 좋음' 방향 학습 |
| KL | \((0.95,0.05)\|(0.75,0.25)\) ≈ 0.495, 동일=0 | KL = "기준 정책에서 벗어난 정도"의 단일 수치 |
| 보상 해킹 sweep | \(\beta=0\): V=−8; \(\beta=100\): V=+0.23 | \(\beta\)가 작으면 보상 해킹, 크면 동설(over-regularization) |
| DPO 손실 | z 0.811→1.658, 손실 0.368→0.174 | BT 손실과 같은 곡선; 보상의 자리에 정책 로그-비율 |
| LoRA/SVD | rank-4 정확, r=3: 33%, r=1: 70%; 7B급 r=8 = 0.39% | "저차원"이 충분한 이유(진짜 rank를 넘으면 정보 손실) |
| SFT 분포 | 0.0011 → 0.0323 (≈30x), CE 0.98 → 0.49 | 포스트트레이닝 = 분포 안의 "선택 기준" 재조정 |

**다음: ML2** — PPO(클리핑 목적함수)의 유도·구현과, 보상 해킹이
실제 강화학습 루프에서 어떻게 관측되는지. (13.3절 본문의 KL 항과
"같은 문제의 다른 얼굴".)